### Benchmarking (running in direct, non-framework mode)

Demonstrates the ability to benchmark llm responses against an given semantic expectation directly. Later functionality will allow you to run in framework mode, in which benchmarks are auto-executed using defined fixtures as parameters.

#### Requires:
- .env file in project directory configured with `OPENAI_API_KEY`, `BASE_URL`, `DEFAULT_EMBEDDING_MODEL` (or using defaults)

In [1]:
import os
os.chdir('..')

import semtest

#### Defined semantic expectation

In [2]:
expectation = "A dog is in the background of the photograph"

In [3]:
def mock_llm_response_prompt1():
    yield from [
        "There's a dog in the background of the photo",
        "In the background of the photo is a dog",
        "There's an animal in the background of the photo and it's a dog."
    ]

mock_llm_response_generator_prompt_1 = mock_llm_response_prompt1()

In [4]:
def mock_llm_response_prompt2():
    yield from [
        "In the background of the photograph there is a furry animal",
        "In the foreground there is a human, and I see a dog in the background of the photograph",
        "There are two dogs in the background of the image"
    ]

mock_llm_response_generator_prompt_2 = mock_llm_response_prompt2()

### Generate basic semantic comparator

In [5]:
from semtest import CosineSimilarity
cosine_similarity = CosineSimilarity(
    semantic_expectation=expectation,
)

#### Decorate the function to act as a benchmark

In [6]:

@semtest.benchmark(
    comparator=cosine_similarity,
    iterations=3,
)
def mock_prompt_benchmark_prompt_1() -> str:
    """A better prompt/temperature/config"""

    # intermediary logic ...
    
    mocked_llm_response = next(mock_llm_response_generator_prompt_1)  # mock llm response

    # user validations ...

    return mocked_llm_response

In [7]:
@semtest.benchmark(
    comparator=cosine_similarity,
    iterations=3
)
def mock_prompt_benchmark_prompt_2() -> str:
    """A slightly worse prompt/temperature/config"""

    # intermediary logic ...
    
    mocked_llm_response = next(mock_llm_response_generator_prompt_2)  # mock llm response

    # user validations ...

    return mocked_llm_response

In [8]:
prompt_1_result: semtest.BenchmarkMetadata = mock_prompt_benchmark_prompt_1()
prompt_2_result: semtest.BenchmarkMetadata = mock_prompt_benchmark_prompt_2()

In [9]:
print(prompt_1_result)
print(prompt_2_result)

func='mock_prompt_benchmark_prompt_1' iterations=3 comparator='cosine_similarity' expectation='A dog is in the background of the photograph' benchmarks=SemanticMetrics(responses=["There's a dog in the background of the photo", 'In the background of the photo is a dog', "There's an animal in the background of the photo and it's a dog."], exceptions=[], semantic_distances=[np.float64(0.8689810275258625), np.float64(0.8314036383391761), np.float64(0.7697098811099436)], mean_semantic_distance=np.float64(0.8233648489916607), median_semantic_distance=np.float64(0.8314036383391761))
func='mock_prompt_benchmark_prompt_2' iterations=3 comparator='cosine_similarity' expectation='A dog is in the background of the photograph' benchmarks=SemanticMetrics(responses=['In the background of the photograph there is a furry animal', 'In the foreground there is a human, and I see a dog in the background of the photograph', 'There are two dogs in the background of the image'], exceptions=[], semantic_distan

In [10]:
print(prompt_1_result.model_dump_json(indent=2))
print(prompt_2_result.model_dump_json(indent=2))

{
  "func": "mock_prompt_benchmark_prompt_1",
  "iterations": 3,
  "comparator": "cosine_similarity",
  "expectation": "A dog is in the background of the photograph",
  "benchmarks": {
    "responses": [
      "There's a dog in the background of the photo",
      "In the background of the photo is a dog",
      "There's an animal in the background of the photo and it's a dog."
    ],
    "exceptions": [],
    "semantic_distances": [
      0.8689810275258625,
      0.8314036383391761,
      0.7697098811099436
    ],
    "mean_semantic_distance": 0.8233648489916607,
    "median_semantic_distance": 0.8314036383391761
  }
}
{
  "func": "mock_prompt_benchmark_prompt_2",
  "iterations": 3,
  "comparator": "cosine_similarity",
  "expectation": "A dog is in the background of the photograph",
  "benchmarks": {
    "responses": [
      "In the background of the photograph there is a furry animal",
      "In the foreground there is a human, and I see a dog in the background of the photograph",
  